In [4]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

model.to(device)
model.eval()
def fariz_gpt(prompt, max_length=100, temperature=0.3, top_k=20):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)
question = "Explain transformer architecture in simple terms"
response = fariz_gpt(question)

print("Fariz-GPT:", response)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Fariz-GPT: Explain transformer architecture in simple terms.

In the above example, the transformer is a singleton with a singleton transformer.

The transformer is a singleton with a singleton transformer.

The transformer is a singleton with a singleton transformer.

The transformer is a singleton with a singleton transformer.

The transformer is a singleton with a singleton transformer.

The transformer is a singleton with a singleton transformer.

The


In [ ]:
# -----------------------------
# 1️⃣ Install required libraries
# -----------------------------
# !pip install torch transformers pdfplumber sentencepiece --quiet

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.optim import AdamW

import pdfplumber
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 2️⃣ Load and preprocess PDF
# -----------------------------
pdf_path = "transformers.pdf"  # your PDF path

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    text = re.sub(r'\s+', ' ', text)
    return text

raw_text = extract_text_from_pdf(pdf_path)

# Split text into small chunks for training
chunk_size = 200  # number of characters per chunk
chunks = [raw_text[i:i+chunk_size] for i in range(0, len(raw_text), chunk_size)]

# -----------------------------
# 3️⃣ Tokenizer & Dataset
# -----------------------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no pad token

class PDFDataset(Dataset):
    def __init__(self, chunks, tokenizer, max_length=200):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        for chunk in chunks:
            encodings = self.tokenizer(chunk, truncation=True, max_length=max_length, padding='max_length')
            self.examples.append(torch.tensor(encodings['input_ids']))
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        x = self.examples[idx]
        return x, x  # input = output for language modeling

dataset = PDFDataset(chunks, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# 4️⃣ Load GPT2 Model for Fine-Tuning
# -----------------------------
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))  # adjust token embeddings
model.to(device)
model.train()

# -----------------------------
# 5️⃣ Optimizer & Loss
# -----------------------------
optimizer = AdamW(model.parameters(), lr=5e-5)

# -----------------------------
# 6️⃣ Fine-Tune GPT
# -----------------------------
epochs = 3  # for demo, increase for better results
for epoch in range(epochs):
    total_loss = 0
    for input_ids, labels in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

# -----------------------------
# 7️⃣ PDF-Grounded Fariz-GPT Inference
# -----------------------------
def fariz_gpt_pdf(prompt, max_length=150, temperature=0.3, top_k=20):
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        output = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        return tokenizer.decode(output[0], skip_special_tokens=True)

# -----------------------------
# 8️⃣ Ask a question
# -----------------------------
question = "Explain transformer architecture in simple terms."
answer = fariz_gpt_pdf(question)
print("Fariz-GPT (PDF):", answer)


In [5]:


import gradio as gr

def chat_interface(question):
    return fariz_gpt_pdf(question)

iface = gr.Interface(fn=chat_interface, inputs="text", outputs="text", title="Fariz-GPT PDF Chatbot")
iface.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
  File "c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
  File "c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\gradio\blocks.py", line 2152, in process_api
    result = await self.call_function(
  File "c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\gradio\blocks.py", line 1629, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
  File "c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\anyio\to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
  File "c:\Users\USER\anaconda3\envs\haar_env\lib\site-packages\anyio\_backends\_asyncio.py", line 2502, in run_sync_in_worker_thread
    re

Using existing dataset file at: .gradio\flagged\dataset1.csv


In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

model.to(device)
model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
def chat_gpt(prompt, max_new_tokens=100, temperature=0.7, top_k=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,  # ✅ FIX
        temperature=temperature,
        top_k=top_k,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response


In [5]:
MAX_CONTEXT_CHARS = 1000
chat_history = chat_history[-MAX_CONTEXT_CHARS:]


In [ ]:
print("🤖 Fariz-GPT Chatbot (type 'exit' to quit)\n")

chat_history = ""

while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Fariz-GPT: Goodbye 👋")
        break

    chat_history += f"You: {user_input}\nFariz-GPT:"

    response = chat_gpt(chat_history, max_new_tokens=100)
    
    answer = response[len(chat_history):].strip()
    print("Fariz-GPT:", answer)

    chat_history += answer + "\n"


🤖 Fariz-GPT Chatbot (type 'exit' to quit)

Fariz-GPT: Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J: 
Fritz-J:
